# Missing Data Analysis
## Sri Lanka Civil Registration and Vital Statistics (CRVS) Data 
### Birth Registration Study — Parity and Its Determinants

---

## 1. Overview and Methodological Rationale

This notebook implements the missing data analysis specified in **Section 3.4.2** of the methodology chapter.

### Theoretical Framework — Rubin's Missing Data Mechanisms (1976)

Per **Section 3.4.2** of the methodology, missing data are classified using Rubin's (1976) typology:

| Mechanism | Definition | Implication |
|---|---|---|
| **MCAR** (Missing Completely At Random) | Missingness is unrelated to any observed or unobserved data | Listwise deletion produces unbiased estimates |
| **MAR** (Missing At Random) | Missingness depends on observed data but not unobserved values | Listwise deletion may introduce bias; multiple imputation preferred |
| **MNAR** (Missing Not At Random) | Missingness depends on unobserved values themselves | All standard methods produce biased estimates; sensitivity analysis required |

### Methodological Decision: Listwise Deletion

> *"Given the very large sample size of the CRVS dataset and the relatively low expected missingness rate for most variables, listwise deletion was adopted as the primary approach for handling missing data. This approach is consistent with practice in large-scale demographic studies (Rutstein & Rojas, 2006; Sterne et al., 2009) and avoids the additional computational complexity and model dependence introduced by multiple imputation."* (Section 3.4.2.3)

### Critical Assumption Verified Here

> *"Listwise deletion produces unbiased estimates under MCAR and approximately unbiased estimates under MAR when the proportion of missing data is small. Patterns of missingness were therefore examined to assess the plausibility of these assumptions."* (Section 3.4.2.3)

### Variables of Particular Concern

Per **Section 3.4.2.1**:
- **Birth Weight** — expected highest missingness; linked to non-hospital delivery (MAR mechanism)
- **Race of Father** — high missingness possible due to administrative reporting practices
- **Marital Status** — variable completeness; cultural sensitivity issues

### References
- Rubin, D. B. (1976). Inference and missing data. *Biometrika*, 63(3), 581–592. https://doi.org/10.1093/biomet/63.3.581
- Little, R. J. A. (1988). A test of missing completely at random for multivariate data with missing values. *Journal of the American Statistical Association*, 83(404), 1198–1202. https://doi.org/10.1080/01621459.1988.10478722
- Little, R. J. A., & Rubin, D. B. (2002). *Statistical analysis with missing data* (2nd ed.). Wiley. https://doi.org/10.1002/9781119013563
- Sterne, J. A. C., White, I. R., Carlin, J. B., Spratt, M., Royston, P., Kenward, M. G., Wood, A. M., & Carpenter, J. R. (2009). Multiple imputation for missing data in epidemiological and clinical research: Potential and pitfalls. *BMJ*, 338, b2393. https://doi.org/10.1136/bmj.b2393
- Rutstein, S. O., & Rojas, G. (2006). *Guide to DHS statistics* (DHS-IV). ORC Macro.

---
## 2. Environment Setup

In [9]:
# ── Core libraries ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Statistical testing ───────────────────────────────────────────────────────
from scipy import stats
from scipy.stats import chi2_contingency, chi2

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Display settings ──────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 150)

# ── Plot aesthetics ───────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

# ── Constants per methodology ─────────────────────────────────────────────────
ALPHA               = 0.05    # significance threshold
MISSING_LOW_PCT     = 5.0     # < 5% considered low (Sterne et al., 2009)
MISSING_MOD_PCT     = 10.0    # 5–10% moderate
MISSING_HIGH_PCT    = 20.0    # > 20% high — careful examination required

print('Environment ready.')
print(f'Significance threshold α = {ALPHA}')
print(f'Missingness severity bands: <{MISSING_LOW_PCT}% low | '
      f'{MISSING_LOW_PCT}–{MISSING_MOD_PCT}% moderate | '
      f'{MISSING_MOD_PCT}–{MISSING_HIGH_PCT}% high | '
      f'>{MISSING_HIGH_PCT}% very high')

Environment ready.
Significance threshold α = 0.05
Missingness severity bands: <5.0% low | 5.0–10.0% moderate | 10.0–20.0% high | >20.0% very high


---
## 3. Load Raw Dataset

Missing data analysis must be conducted on the **raw dataset before any cleaning or exclusions**, as specified in the methodology. The full pattern of missingness is the diagnostic input for the listwise deletion decision.

In [13]:
from pathlib import Path

In [16]:
# Define paths
DATA_PATH = Path("../data/raw")

In [17]:
# Load data
df = pd.read_excel(DATA_PATH / "2000_Birth_Final.xlsx")

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# INSTRUCTION: Replace FILE_PATH with the path to your RAW CRVS dataset
# (i.e., the dataset before any outlier exclusion or cleaning was applied).
# This is the dataset described in Section 3.2 of the methodology.
# ─────────────────────────────────────────────────────────────────────────────


n_total = len(df)
print(f'Dataset loaded: {n_total:,} records × {df.shape[1]} variables')
print()
print('First three records:')
df.head(3)

Dataset loaded: 347,749 records × 14 variables

First three records:


,Registered_Year,Registered_Month,Registered_District,Birh_Year_2.0,Birh_Year,Birth_Month,Gender,Hospital or Not,Birth_Order 2.0,Birth_Order,Age of Mother,Marital_Status,District_of_Mother,Race_of_Mother
0,2000,April,Puttalam,2000,2000,January,Male,Not in Hospital,1,First,25,Married,Kurunagala,Sinhalese
1,2000,April,Puttalam,2000,2000,January,Male,Not in Hospital,1,First,25,Married,Puttalam,Sinhalese
2,2000,April,Puttalam,2000,2000,February,Male,Not in Hospital,1,First,22,Married,Puttalam,Sinhalese


---
## 4. Variable Mapping and Standardisation

Variable names are mapped to standardised labels for consistent reporting. Common missing value codes used in CRVS data (e.g., empty strings, 99, 999, 'Unknown', 'Not Stated') are converted to proper NaN.

In [ ]:
# ── Column name mapping — update to match your dataset ────────────────────────
VARIABLES = {
    'Age of Mother'          : 'Age of Mother',
    'Marital_Status'         : 'Marital Status',
    'Race_of_Mother'         : 'Race of Mother',
    'Race_of_Father'         : 'Race of Father',
    'Gender'                 : 'Sex of Child',
    'Hospital or Not'        : 'Place of Delivery',
    'Multiple_Birth_Status'  : 'Multiple Birth Status',
    'Birth_Weight (grams)'   : 'Birth Weight',
    'Birth_Order'            : 'Birth Order (Parity)',
}

# Filter to columns present in dataset
available_vars = {k: v for k, v in VARIABLES.items() if k in df.columns}
missing_vars   = {k: v for k, v in VARIABLES.items() if k not in df.columns}

print(f'Variables found ({len(available_vars)}):')
for col, label in available_vars.items():
    print(f'  ✓ "{col}" → {label}')
if missing_vars:
    print(f'\nVariables NOT found ({len(missing_vars)}):')
    for col, label in missing_vars.items():
        print(f'  ✗ "{col}" — update the VARIABLES mapping')

In [ ]:
# ── Standardise missing value codes ────────────────────────────────────────────
# CRVS data may encode missingness as: '', ' ', 'Unknown', 'Not Stated',
# 'NA', 'N/A', '99', '999', '9999' (depending on field type).

MISSING_CODES_TEXT    = ['', ' ', 'NA', 'N/A', 'Unknown', 'Not Stated',
                          'NotStated', 'Not stated', 'unknown', 'NaN', 'nan']
MISSING_CODES_NUMERIC = [99, 999, 9999, -1, -99]

df_raw = df.copy()

# Replace text-coded missing values
df_raw = df_raw.replace(MISSING_CODES_TEXT, np.nan)

# Replace numeric-coded missing in continuous fields only (preserves valid data)
for col in ['Age of Mother', 'Birth_Weight (grams)']:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
        # Numeric sentinel codes only valid if outside plausible range
        if col == 'Age of Mother':
            df_raw.loc[df_raw[col] == 99, col]  = np.nan
            df_raw.loc[df_raw[col] == 999, col] = np.nan
        elif col == 'Birth_Weight (grams)':
            df_raw.loc[df_raw[col] == 9999, col] = np.nan

# Strip whitespace from text fields
for col in df_raw.select_dtypes(include='object').columns:
    df_raw[col] = df_raw[col].astype(str).str.strip()
    df_raw.loc[df_raw[col].isin(['', 'nan', 'NaN']), col] = np.nan

print('Missing value standardisation complete.')
print(f'Total records: {len(df_raw):,}')

---
## 5. Step 1 — Univariate Missingness Quantification (Section 3.4.2.1)

The first step is to quantify the proportion of missing values for every variable. Per the methodology, the rate of missingness for each variable is reported and assessed against the severity bands defined by Sterne et al. (2009).

### Reference
- Sterne, J. A. C., et al. (2009). *BMJ*, 338, b2393.

In [ ]:
def classify_missingness(pct):
    """Classify missingness severity per Sterne et al. (2009) framework."""
    if pct == 0:
        return 'Complete (0%)'
    elif pct < MISSING_LOW_PCT:
        return f'Low (< {MISSING_LOW_PCT}%) — listwise deletion safe'
    elif pct < MISSING_MOD_PCT:
        return f'Moderate ({MISSING_LOW_PCT}–{MISSING_MOD_PCT}%) — examine pattern'
    elif pct < MISSING_HIGH_PCT:
        return f'High ({MISSING_MOD_PCT}–{MISSING_HIGH_PCT}%) — careful examination'
    else:
        return f'Very high (> {MISSING_HIGH_PCT}%) — sensitivity analysis required'


# ── Build missingness table ───────────────────────────────────────────────────
miss_rows = []
for col, label in available_vars.items():
    n_miss   = df_raw[col].isna().sum()
    pct_miss = (n_miss / n_total) * 100
    miss_rows.append({
        'Variable'           : label,
        'Column'             : col,
        'N missing'          : n_miss,
        'N present'          : n_total - n_miss,
        'Missing (%)'        : round(pct_miss, 4),
        'Severity'           : classify_missingness(pct_miss)
    })

miss_df = pd.DataFrame(miss_rows).sort_values('Missing (%)', ascending=False)

print('─── Univariate Missingness Summary (Section 3.4.2.1) ───────────────────')
print(miss_df.to_string(index=False))

In [ ]:
# ── Visualisation: missingness bar chart ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, max(5, len(miss_df)*0.45)))

# Colour by severity
def severity_colour(pct):
    if pct == 0:                  return '#cccccc'
    elif pct < MISSING_LOW_PCT:   return '#2ca02c'
    elif pct < MISSING_MOD_PCT:   return '#1f77b4'
    elif pct < MISSING_HIGH_PCT:  return '#ff7f0e'
    else:                         return '#d62728'

colours = [severity_colour(p) for p in miss_df['Missing (%)']]

bars = ax.barh(miss_df['Variable'], miss_df['Missing (%)'],
               color=colours, edgecolor='white', alpha=0.85)

# Threshold lines
ax.axvline(MISSING_LOW_PCT,  color='#1f77b4', linestyle=':', lw=1.2, alpha=0.7)
ax.axvline(MISSING_MOD_PCT,  color='#ff7f0e', linestyle=':', lw=1.2, alpha=0.7)
ax.axvline(MISSING_HIGH_PCT, color='#d62728', linestyle='--', lw=1.5, alpha=0.7)

# Value labels
for bar, pct in zip(bars, miss_df['Missing (%)']):
    ax.text(pct + 0.3, bar.get_y() + bar.get_height()/2,
            f'{pct:.2f}%', va='center', fontsize=9)

# Legend
patches = [
    mpatches.Patch(color='#cccccc', alpha=0.85, label='Complete (0%)'),
    mpatches.Patch(color='#2ca02c', alpha=0.85, label=f'Low (< {MISSING_LOW_PCT}%)'),
    mpatches.Patch(color='#1f77b4', alpha=0.85, label=f'Moderate ({MISSING_LOW_PCT}–{MISSING_MOD_PCT}%)'),
    mpatches.Patch(color='#ff7f0e', alpha=0.85, label=f'High ({MISSING_MOD_PCT}–{MISSING_HIGH_PCT}%)'),
    mpatches.Patch(color='#d62728', alpha=0.85, label=f'Very high (> {MISSING_HIGH_PCT}%)'),
]
ax.legend(handles=patches, fontsize=9, loc='lower right')

ax.set_title('Missingness Rate by Variable — Severity Classification\n'
             '(Section 3.4.2.1 — Sterne et al., 2009)', fontweight='bold')
ax.set_xlabel('Percentage Missing (%)')
ax.set_xlim(0, max(miss_df['Missing (%)'].max() * 1.15, MISSING_HIGH_PCT + 5))
plt.tight_layout()
plt.savefig('fig1_missingness_by_variable.png', bbox_inches='tight')
plt.show()
print('Figure saved: fig1_missingness_by_variable.png')

---
## 6. Step 2 — Missingness Pattern Visualisation (Section 3.4.2.2)

Per **Section 3.4.2.2**, the *pattern* of missingness across variables is examined to detect whether records tend to be missing on multiple variables simultaneously (block missingness) or whether missingness occurs independently across variables.

### Reference
- Little, R. J. A., & Rubin, D. B. (2002). *Statistical analysis with missing data* (2nd ed.). Wiley.

In [ ]:
# ── Build binary missingness indicator matrix ─────────────────────────────────
var_cols   = list(available_vars.keys())
var_labels = list(available_vars.values())

miss_indicator = df_raw[var_cols].isna().astype(int)
miss_indicator.columns = var_labels

print(f'Missingness indicator matrix shape: {miss_indicator.shape}')
print('\nFirst 10 rows of indicator matrix (1 = missing, 0 = present):')
print(miss_indicator.head(10).to_string())

In [ ]:
# ── Heatmap visualisation of missingness pattern ──────────────────────────────
# Sample for visualisation if dataset very large (>5,000 records)
sample_size = min(2000, len(miss_indicator))
miss_sample = miss_indicator.sample(n=sample_size, random_state=42)
miss_sample = miss_sample.sort_values(by=list(miss_sample.columns))

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(miss_sample.T, cmap=['#2ca02c', '#d62728'], cbar=False,
            yticklabels=True, xticklabels=False, ax=ax,
            linewidths=0, linecolor=None)

ax.set_title(f'Missing Data Pattern — Heatmap of {sample_size:,} Random Records\n'
             'Green = Present | Red = Missing | Sorted by missingness pattern',
             fontweight='bold')
ax.set_xlabel('Records (sorted)')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('fig2_missingness_pattern_heatmap.png', bbox_inches='tight')
plt.show()
print('Figure saved: fig2_missingness_pattern_heatmap.png')

In [ ]:
# ── Identify and quantify unique missingness patterns ─────────────────────────
pattern_str = miss_indicator.astype(str).agg(''.join, axis=1)
pattern_counts = pattern_str.value_counts().head(15)

# Build readable pattern table
pattern_rows = []
for pattern_code, count in pattern_counts.items():
    missing_vars_in_pattern = [
        var_labels[i] for i, c in enumerate(pattern_code) if c == '1'
    ]
    if not missing_vars_in_pattern:
        pattern_desc = 'COMPLETE — no variables missing'
    else:
        pattern_desc = ' + '.join(missing_vars_in_pattern)
    pattern_rows.append({
        'Pattern code'   : pattern_code,
        'N records'      : count,
        '% of dataset'   : round(100*count/n_total, 4),
        'Variables missing': pattern_desc
    })

pattern_df = pd.DataFrame(pattern_rows)
print('─── Top 15 Missingness Patterns (Section 3.4.2.2) ───────────────────────')
print('(Pattern code: 1=missing, 0=present, in column order)')
print()
for col in var_labels:
    print(f'  Position {var_labels.index(col)}: {col}')
print()
print(pattern_df.to_string(index=False))

In [ ]:
# ── Records-with-N-missing chart ──────────────────────────────────────────────
n_missing_per_record = miss_indicator.sum(axis=1)
n_missing_distribution = n_missing_per_record.value_counts().sort_index()
n_missing_pct = (n_missing_distribution / n_total * 100).round(4)

summary_dist = pd.DataFrame({
    'N variables missing': n_missing_distribution.index,
    'N records'         : n_missing_distribution.values,
    '% of dataset'      : n_missing_pct.values
})
summary_dist['Cumulative %'] = summary_dist['% of dataset'].cumsum().round(4)

print('─── Distribution: Number of Variables Missing per Record ────────────────')
print(summary_dist.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
colours_dist = ['#2ca02c'] + ['#1f77b4']*2 + ['#ff7f0e']*2 + ['#d62728']*20
colours_dist = colours_dist[:len(summary_dist)]

bars = ax.bar(summary_dist['N variables missing'], summary_dist['N records'],
              color=colours_dist, edgecolor='white', alpha=0.85)

for bar, pct in zip(bars, summary_dist['% of dataset']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + summary_dist['N records'].max()*0.01,
            f'{pct:.2f}%', ha='center', va='bottom', fontsize=9)

ax.set_title('Distribution: Number of Variables Missing per Record\n'
             '(Section 3.4.2.2 — Block missingness assessment)',
             fontweight='bold')
ax.set_xlabel('Number of Variables Missing')
ax.set_ylabel('Number of Records')
ax.set_xticks(summary_dist['N variables missing'])
plt.tight_layout()
plt.savefig('fig3_n_missing_per_record.png', bbox_inches='tight')
plt.show()
print('Figure saved: fig3_n_missing_per_record.png')

---
## 7. Step 3 — Missingness Correlation Matrix

If missingness on one variable correlates strongly with missingness on another, this suggests a shared mechanism (e.g., a non-hospital delivery may produce missingness on both Birth Weight and Hospital Indicator simultaneously). Per **Section 3.4.2.2**, this informs the MAR/MNAR assessment.

### Reference
- Little, R. J. A., & Rubin, D. B. (2002). *Statistical analysis with missing data* (2nd ed.). Wiley.

In [ ]:
# Drop variables with no missingness (correlation undefined)
miss_with_var = miss_indicator.loc[:, miss_indicator.sum() > 0]

if miss_with_var.shape[1] >= 2:
    corr_miss = miss_with_var.corr()

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr_miss, annot=True, fmt='.3f', cmap='RdYlGn_r',
                center=0, vmin=-1, vmax=1, square=True,
                linewidths=0.5, linecolor='white', cbar_kws={'label': 'Correlation'})
    ax.set_title('Correlation of Missingness Indicators\n'
                 '(High positive = shared missingness mechanism)',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig4_missingness_correlation.png', bbox_inches='tight')
    plt.show()
    print('Figure saved: fig4_missingness_correlation.png')
    print()
    print('─── Strong missingness correlations (|r| > 0.30) ────────────────────')
    found = False
    for i in range(len(corr_miss)):
        for j in range(i+1, len(corr_miss)):
            r = corr_miss.iloc[i, j]
            if abs(r) > 0.30:
                found = True
                print(f'  {corr_miss.index[i]:<25} ↔ {corr_miss.columns[j]:<25}: r = {r:+.3f}')
    if not found:
        print('  No correlations exceed |r| = 0.30 — missingness largely independent across variables.')
else:
    print('Insufficient variables with missingness for correlation matrix.')

---
## 8. Step 4 — Little's MCAR Test

Little's (1988) test of MCAR formally tests the null hypothesis that the data are Missing Completely At Random. A non-significant p-value (p ≥ 0.05) is consistent with MCAR; a significant p-value rejects MCAR and implies MAR or MNAR.

**Caveat per methodology** (Section 3.4.2):
> *"Little's test is sensitive to large sample size and may reject MCAR even for small departures. Results are interpreted alongside pattern visualisation rather than relied upon mechanically."*

### Reference
- Little, R. J. A. (1988). *Journal of the American Statistical Association*, 83(404), 1198–1202.

In [ ]:
def littles_mcar_test(df_numeric):
    """Implement Little's (1988) MCAR test for multivariate continuous data.

    Parameters
    ----------
    df_numeric : pd.DataFrame
        Numeric dataframe with potential missing values (NaN).

    Returns
    -------
    dict with chi2 statistic, df, p-value, and conclusion.

    References
    ----------
    Little, R. J. A. (1988). A test of missing completely at random for
    multivariate data with missing values. Journal of the American Statistical
    Association, 83(404), 1198–1202.
    """
    # Drop columns that are completely missing or completely observed
    df_n = df_numeric.copy()
    keep_cols = [c for c in df_n.columns
                 if 0 < df_n[c].isna().sum() < len(df_n)]
    df_n = df_n[keep_cols].astype(float)

    if df_n.shape[1] < 2:
        return {'chi2': np.nan, 'df': 0, 'p_value': np.nan,
                'conclusion': 'Insufficient variables with missingness'}

    # Global mean and covariance using all available pairwise data
    global_mean = df_n.mean()
    global_cov  = df_n.cov()

    # Identify unique missingness patterns
    pattern_idx = df_n.isna().astype(int).astype(str).agg(''.join, axis=1)
    unique_patterns = pattern_idx.unique()

    chi2_stat = 0.0
    total_df  = 0

    for pattern in unique_patterns:
        # Mask: which variables are observed in this pattern
        observed_mask = np.array([c == '0' for c in pattern])
        if observed_mask.sum() == 0:
            continue

        observed_cols = df_n.columns[observed_mask].tolist()
        subset = df_n.loc[pattern_idx == pattern, observed_cols]
        n_j = len(subset)
        if n_j == 0:
            continue

        # Pattern mean
        pattern_mean = subset.mean()
        diff = pattern_mean - global_mean[observed_cols]

        # Sub-covariance for observed variables
        sub_cov = global_cov.loc[observed_cols, observed_cols].values
        try:
            inv_cov = np.linalg.pinv(sub_cov)
        except np.linalg.LinAlgError:
            continue

        # Mahalanobis-type contribution
        chi2_stat += n_j * float(diff.values @ inv_cov @ diff.values)
        total_df  += observed_mask.sum()

    # Degrees of freedom: sum(p_j) − p
    total_df = max(total_df - df_n.shape[1], 1)
    p_val   = 1 - chi2.cdf(chi2_stat, total_df)

    if p_val < 0.05:
        conclusion = ('REJECT MCAR (p < 0.05) — missingness is likely MAR or MNAR. '
                     'Examine pattern visualisations to assess MAR plausibility.')
    else:
        conclusion = 'FAIL TO REJECT MCAR — consistent with random missingness.'

    return {'chi2': round(chi2_stat, 3),
            'df': int(total_df),
            'p_value': p_val,
            'conclusion': conclusion,
            'variables_used': keep_cols}


# ── Apply Little's MCAR test to numeric variables ─────────────────────────────
numeric_cols = [c for c in ['Age of Mother', 'Birth_Weight (grams)']
                if c in df_raw.columns]

if len(numeric_cols) >= 2:
    df_num = df_raw[numeric_cols].apply(pd.to_numeric, errors='coerce')
    mcar_result = littles_mcar_test(df_num)

    print("─── Little's (1988) MCAR Test ──────────────────────────────────────────")
    print(f"Variables tested  : {mcar_result['variables_used']}")
    print(f"χ² statistic      : {mcar_result['chi2']}")
    print(f"Degrees of freedom: {mcar_result['df']}")
    print(f"p-value           : "
          f"{'< 0.001' if mcar_result['p_value'] < 0.001 else f'{mcar_result[chr(34)+chr(112)+chr(95)+chr(118)+chr(97)+chr(108)+chr(117)+chr(101)+chr(34)]:.4f}'}")
    print(f"Conclusion        : {mcar_result['conclusion']}")
    print()
    print('Caveat: Little\'s test is sensitive to sample size and may reject MCAR')
    print('for trivial departures in large datasets. Interpret alongside pattern')
    print('visualisations (Sections 3.4.2.2). (Sterne et al., 2009)')
else:
    print('Insufficient numeric variables for Little\'s test (need ≥ 2 with missingness).')

---
## 9. Step 5 — MAR Mechanism Testing (Section 3.4.2.2)

Per the methodology, the MAR mechanism is most plausible when missingness on a variable can be explained by *observed* values of other variables. The most important test for this study is whether **Birth Weight missingness is explained by delivery setting** (hospital vs non-hospital), as expected from clinical practice in Sri Lanka in 2000.

Each variable with non-zero missingness is tested:
- **For categorical predictors** → chi-square test on missingness indicator
- **For continuous predictors** → t-test or Mann-Whitney U on missingness groups

### Reference
- Sterne, J. A. C., et al. (2009). *BMJ*, 338, b2393.
- Rubin, D. B. (1976). *Biometrika*, 63(3), 581–592.

In [ ]:
def test_mar_categorical(df, target_col, predictor_col, alpha=0.05):
    """Test whether missingness in target_col depends on a categorical predictor."""
    df_test = df.copy()
    df_test['_missing'] = df_test[target_col].isna().astype(int)
    df_test = df_test.dropna(subset=[predictor_col])
    if df_test['_missing'].sum() == 0 or df_test['_missing'].sum() == len(df_test):
        return None
    ct = pd.crosstab(df_test[predictor_col], df_test['_missing'])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return None
    chi2_stat, p_val, dof, _ = chi2_contingency(ct)
    return {'test': 'Chi-square',
            'statistic': round(chi2_stat, 3),
            'df': int(dof),
            'p_value': p_val,
            'significant': p_val < alpha}


def test_mar_continuous(df, target_col, predictor_col, alpha=0.05):
    """Test whether missingness in target_col depends on a continuous predictor."""
    df_test = df.copy()
    df_test[predictor_col] = pd.to_numeric(df_test[predictor_col], errors='coerce')
    df_test['_missing'] = df_test[target_col].isna().astype(int)
    df_test = df_test.dropna(subset=[predictor_col])
    grp_missing = df_test.loc[df_test['_missing']==1, predictor_col]
    grp_present = df_test.loc[df_test['_missing']==0, predictor_col]
    if len(grp_missing) < 2 or len(grp_present) < 2:
        return None
    # Mann-Whitney U (robust to non-normality)
    u_stat, p_val = stats.mannwhitneyu(grp_missing, grp_present, alternative='two-sided')
    return {'test': 'Mann-Whitney U',
            'statistic': round(u_stat, 3),
            'mean_missing': round(grp_missing.mean(), 2),
            'mean_present': round(grp_present.mean(), 2),
            'p_value': p_val,
            'significant': p_val < alpha}


# ── Apply MAR tests for each variable with missingness ────────────────────────
mar_rows = []
categorical_predictors = ['Marital_Status', 'Race_of_Mother', 'Gender',
                          'Hospital or Not', 'Multiple_Birth_Status', 'Birth_Order']
continuous_predictors  = ['Age of Mother', 'Birth_Weight (grams)']

vars_with_missing = miss_df.loc[miss_df['Missing (%)'] > 0, 'Column'].tolist()

for tgt_col in vars_with_missing:
    tgt_label = available_vars.get(tgt_col, tgt_col)

    for pred_col in categorical_predictors:
        if pred_col == tgt_col or pred_col not in df_raw.columns:
            continue
        res = test_mar_categorical(df_raw, tgt_col, pred_col, alpha=ALPHA)
        if res is None:
            continue
        mar_rows.append({
            'Missing variable'  : tgt_label,
            'Tested against'    : available_vars.get(pred_col, pred_col),
            'Predictor type'    : 'Categorical',
            'Test'              : res['test'],
            'Statistic'         : res['statistic'],
            'df'                : res['df'],
            'p-value'           : ('< 0.001' if res['p_value'] < 0.001
                                    else f"{res['p_value']:.4f}"),
            'Significant association': 'YES — MAR consistent' if res['significant']
                                       else 'No'
        })

    for pred_col in continuous_predictors:
        if pred_col == tgt_col or pred_col not in df_raw.columns:
            continue
        res = test_mar_continuous(df_raw, tgt_col, pred_col, alpha=ALPHA)
        if res is None:
            continue
        mar_rows.append({
            'Missing variable'  : tgt_label,
            'Tested against'    : available_vars.get(pred_col, pred_col),
            'Predictor type'    : 'Continuous',
            'Test'              : res['test'],
            'Statistic'         : res['statistic'],
            'df'                : '—',
            'p-value'           : ('< 0.001' if res['p_value'] < 0.001
                                    else f"{res['p_value']:.4f}"),
            'Significant association': 'YES — MAR consistent' if res['significant']
                                       else 'No'
        })

if mar_rows:
    mar_df = pd.DataFrame(mar_rows)
    print('─── MAR Mechanism Testing — Each Missing Variable vs Predictors ─────────')
    print('(Section 3.4.2.2 — Sterne et al., 2009)')
    print()
    print(mar_df.to_string(index=False))
else:
    print('No variables with missingness available for MAR testing.')

---
## 10. Step 6 — Critical Test: Birth Weight Missingness × Delivery Setting

This is the **single most important MAR test** for this study, as discussed explicitly in **Section 3.4.2.2**:

> *"Birth weight is expected to have higher missingness among non-hospital deliveries because accurate measurement requires clinical weighing equipment unavailable in many home delivery settings. If the data confirm this pattern, missingness on birth weight is plausibly MAR conditional on delivery setting, supporting the validity of listwise deletion when delivery setting is included in the model."*

In [ ]:
BW_COL   = 'Birth_Weight (grams)'
HOSP_COL = 'Hospital or Not'

if BW_COL in df_raw.columns and HOSP_COL in df_raw.columns:
    df_test = df_raw[[BW_COL, HOSP_COL]].copy()
    df_test['BW_missing'] = df_test[BW_COL].isna().astype(int)
    df_test = df_test.dropna(subset=[HOSP_COL])

    # Cross-tab: missingness by delivery setting
    bw_miss_table = df_test.groupby(HOSP_COL).agg(
        N_total   = ('BW_missing', 'count'),
        N_missing = ('BW_missing', 'sum'),
    ).reset_index()
    bw_miss_table['N_present']    = bw_miss_table['N_total'] - bw_miss_table['N_missing']
    bw_miss_table['Missing (%)']  = (
        bw_miss_table['N_missing'] / bw_miss_table['N_total'] * 100
    ).round(3)

    print('─── Birth Weight Missingness by Delivery Setting ────────────────────────')
    print('(Section 3.4.2.2 — Critical MAR mechanism test)')
    print()
    print(bw_miss_table.to_string(index=False))

    # Chi-square test
    ct_bw = pd.crosstab(df_test[HOSP_COL], df_test['BW_missing'])
    chi2_stat, p_val, dof, expected = chi2_contingency(ct_bw)
    n_total_test = ct_bw.values.sum()
    cramers_v_bw = np.sqrt(chi2_stat / (n_total_test * (min(ct_bw.shape)-1)))

    print()
    print(f'χ² statistic   : {chi2_stat:.3f}')
    print(f'df             : {dof}')
    print(f'p-value        : {"< 0.001" if p_val < 0.001 else f"{p_val:.4f}"}')
    print(f"Cramér's V     : {cramers_v_bw:.4f}")
    print()
    if p_val < ALPHA:
        print('CONCLUSION: Birth weight missingness IS significantly associated with')
        print('delivery setting → MAR mechanism CONFIRMED, consistent with the')
        print('methodology hypothesis (Section 3.4.2.2). Listwise deletion remains')
        print('valid when delivery setting is included in regression models.')
    else:
        print('CONCLUSION: No significant association detected. MAR mechanism via')
        print('delivery setting NOT supported in this dataset — sensitivity analysis')
        print('using multiple imputation may be warranted (Sterne et al., 2009).')
else:
    print(f'Required columns not found: "{BW_COL}" and/or "{HOSP_COL}".')

In [ ]:
# ── Visualisation: BW missingness by delivery setting ─────────────────────────
if BW_COL in df_raw.columns and HOSP_COL in df_raw.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel A: Bar chart of % missing by setting
    colours_p = ['#d62728' if 'No' in str(s) or 'Not' in str(s) else '#2ca02c'
                  for s in bw_miss_table[HOSP_COL]]
    bars = axes[0].bar(bw_miss_table[HOSP_COL].astype(str),
                       bw_miss_table['Missing (%)'],
                       color=colours_p, edgecolor='white', alpha=0.85)
    for bar, pct in zip(bars, bw_miss_table['Missing (%)']):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + bw_miss_table['Missing (%)'].max()*0.02,
                     f'{pct:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[0].set_title('Birth Weight Missingness Rate\nby Delivery Setting (MAR test)',
                      fontweight='bold')
    axes[0].set_xlabel('Delivery Setting')
    axes[0].set_ylabel('Birth Weight Missing (%)')
    axes[0].set_ylim(0, max(bw_miss_table['Missing (%)'].max()*1.25, 5))

    # Panel B: Stacked bar — present vs missing
    plot_df = bw_miss_table[[HOSP_COL, 'N_present', 'N_missing']].set_index(HOSP_COL)
    plot_df = plot_df.div(plot_df.sum(axis=1), axis=0) * 100
    plot_df.columns = ['Present', 'Missing']
    plot_df.plot(kind='bar', stacked=True, ax=axes[1],
                 color=['#2ca02c', '#d62728'], edgecolor='white', alpha=0.85)
    axes[1].set_title('Birth Weight Status by Delivery Setting\n(Stacked %)', fontweight='bold')
    axes[1].set_xlabel('Delivery Setting')
    axes[1].set_ylabel('Percentage of Records (%)')
    axes[1].legend(title='Birth Weight', loc='center left', bbox_to_anchor=(1, 0.5))
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

    plt.tight_layout()
    plt.savefig('fig5_bw_missingness_by_setting.png', bbox_inches='tight')
    plt.show()
    print('Figure saved: fig5_bw_missingness_by_setting.png')

---
## 11. Step 7 — Apply Listwise Deletion and Build Analytical Sample

Per **Section 3.4.2.3**, listwise deletion is applied to the full set of analytical variables. Records missing on any predictor or outcome are removed.

In [ ]:
# ── Define analytical variable set (regression-relevant) ──────────────────────
# Race_of_Father is descriptive only — excluded from analytical variable set
# per the methodology variable selection.
ANALYTICAL_VARS = [c for c in [
    'Age of Mother',
    'Marital_Status',
    'Race_of_Mother',
    'Gender',
    'Hospital or Not',
    'Multiple_Birth_Status',
    'Birth_Weight (grams)',
    'Birth_Order'
] if c in df_raw.columns]

print(f'Analytical variable set ({len(ANALYTICAL_VARS)} variables):')
for c in ANALYTICAL_VARS:
    print(f'  • {c}')

# ── Listwise deletion ─────────────────────────────────────────────────────────
df_complete = df_raw.dropna(subset=ANALYTICAL_VARS).copy()

n_after  = len(df_complete)
n_lost   = n_total - n_after
pct_lost = (n_lost / n_total) * 100

print()
print('─── Listwise Deletion Results (Section 3.4.2.3) ─────────────────────────')
print(f'Records before listwise deletion : {n_total:>10,}')
print(f'Records after  listwise deletion : {n_after:>10,}')
print(f'Records dropped                  : {n_lost:>10,}  ({pct_lost:.2f}%)')
print()
if pct_lost < MISSING_LOW_PCT:
    print(f'✓ Loss < {MISSING_LOW_PCT}% — listwise deletion is highly defensible.')
elif pct_lost < MISSING_HIGH_PCT:
    print(f'⚠ Loss {MISSING_LOW_PCT}–{MISSING_HIGH_PCT}% — moderate. Compare characteristics of')
    print(f'  retained vs dropped records (Step 8) to confirm bias is minimal.')
else:
    print(f'⚠⚠ Loss > {MISSING_HIGH_PCT}% — substantial. Consider multiple imputation as')
    print(f'   sensitivity analysis (Sterne et al., 2009).')

---
## 12. Step 8 — Sensitivity Analysis: Retained vs Dropped Records

Per **Section 3.4.2.4**, characteristics of retained records are compared with those dropped by listwise deletion to verify that the analytical sample remains representative.

### Reference
- Little, R. J. A., & Rubin, D. B. (2002). *Statistical analysis with missing data* (2nd ed.). Wiley.
- Sterne, J. A. C., et al. (2009). *BMJ*, 338, b2393.

In [ ]:
# ── Build flag: retained (1) vs dropped (0) ───────────────────────────────────
df_raw['_retained'] = (~df_raw[ANALYTICAL_VARS].isna().any(axis=1)).astype(int)

# ── Compare key descriptive variables ─────────────────────────────────────────
compare_rows = []

# Continuous: Age of Mother, Birth Weight
for col in ['Age of Mother', 'Birth_Weight (grams)']:
    if col not in df_raw.columns:
        continue
    vals = pd.to_numeric(df_raw[col], errors='coerce')
    retained_vals = vals[df_raw['_retained']==1].dropna()
    dropped_vals  = vals[df_raw['_retained']==0].dropna()
    if len(retained_vals) < 2 or len(dropped_vals) < 2:
        continue
    u_stat, p_val = stats.mannwhitneyu(retained_vals, dropped_vals,
                                       alternative='two-sided')
    compare_rows.append({
        'Variable'           : available_vars.get(col, col),
        'Type'               : 'Continuous',
        'Retained (mean)'    : f'{retained_vals.mean():.2f}',
        'Dropped (mean)'     : f'{dropped_vals.mean():.2f}',
        'Retained (median)'  : f'{retained_vals.median():.1f}',
        'Dropped (median)'   : f'{dropped_vals.median():.1f}',
        'Test'               : 'Mann-Whitney U',
        'p-value'            : ('< 0.001' if p_val < 0.001 else f'{p_val:.4f}'),
        'Different?'         : 'YES' if p_val < ALPHA else 'No'
    })

# Categorical: chi-square comparing distributions
for col in ['Marital_Status', 'Race_of_Mother', 'Gender',
            'Hospital or Not', 'Multiple_Birth_Status', 'Birth_Order']:
    if col not in df_raw.columns:
        continue
    sub = df_raw[[col, '_retained']].dropna(subset=[col])
    if sub[col].nunique() < 2 or sub['_retained'].nunique() < 2:
        continue
    ct = pd.crosstab(sub[col], sub['_retained'])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        continue
    chi2_stat, p_val, dof, _ = chi2_contingency(ct)
    # Mode of each group
    mode_ret  = sub.loc[sub['_retained']==1, col].mode().iloc[0] \
                if (sub['_retained']==1).any() else 'N/A'
    mode_drop = sub.loc[sub['_retained']==0, col].mode().iloc[0] \
                if (sub['_retained']==0).any() else 'N/A'
    compare_rows.append({
        'Variable'           : available_vars.get(col, col),
        'Type'               : 'Categorical',
        'Retained (mean)'    : f'mode: {mode_ret}',
        'Dropped (mean)'     : f'mode: {mode_drop}',
        'Retained (median)'  : '—',
        'Dropped (median)'   : '—',
        'Test'               : 'Chi-square',
        'p-value'            : ('< 0.001' if p_val < 0.001 else f'{p_val:.4f}'),
        'Different?'         : 'YES' if p_val < ALPHA else 'No'
    })

compare_df = pd.DataFrame(compare_rows)
print('─── Retained vs Dropped Records — Sensitivity Comparison (Section 3.4.2.4) ─')
print()
print(compare_df.to_string(index=False))
print()
print('Interpretation: Significant differences are EXPECTED in large datasets')
print('and do not necessarily invalidate listwise deletion. The magnitude of')
print('difference (descriptive statistics) is what matters substantively.')
print('(Sterne et al., 2009; Little & Rubin, 2002)')

---
## 13. Step 9 — Distribution Comparison Plots

In [ ]:
# ── Compare distributions of retained vs dropped — continuous variables ───────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'Age of Mother' in df_raw.columns:
    age_vals = pd.to_numeric(df_raw['Age of Mother'], errors='coerce')
    retained = age_vals[df_raw['_retained']==1].dropna()
    dropped  = age_vals[df_raw['_retained']==0].dropna()
    if len(dropped) > 0:
        axes[0].hist(retained, bins=40, alpha=0.6, color='#2ca02c',
                     label=f'Retained (n={len(retained):,})', density=True, edgecolor='white')
        axes[0].hist(dropped, bins=40, alpha=0.6, color='#d62728',
                     label=f'Dropped  (n={len(dropped):,})', density=True, edgecolor='white')
    else:
        axes[0].hist(retained, bins=40, color='#2ca02c', edgecolor='white', density=True)
    axes[0].set_title('Age of Mother — Retained vs Dropped', fontweight='bold')
    axes[0].set_xlabel('Maternal Age (years)')
    axes[0].set_ylabel('Density')
    axes[0].legend(fontsize=9)

if 'Birth_Weight (grams)' in df_raw.columns:
    bw_vals = pd.to_numeric(df_raw['Birth_Weight (grams)'], errors='coerce')
    retained = bw_vals[df_raw['_retained']==1].dropna()
    dropped  = bw_vals[df_raw['_retained']==0].dropna()
    if len(dropped) > 0:
        axes[1].hist(retained, bins=40, alpha=0.6, color='#2ca02c',
                     label=f'Retained (n={len(retained):,})', density=True, edgecolor='white')
        axes[1].hist(dropped, bins=40, alpha=0.6, color='#d62728',
                     label=f'Dropped  (n={len(dropped):,})', density=True, edgecolor='white')
    else:
        axes[1].hist(retained, bins=40, color='#2ca02c', edgecolor='white', density=True)
    axes[1].axvline(2500, color='black', linestyle=':', lw=1.3, label='LBW threshold')
    axes[1].set_title('Birth Weight — Retained vs Dropped', fontweight='bold')
    axes[1].set_xlabel('Birth Weight (grams)')
    axes[1].set_ylabel('Density')
    axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig6_distribution_retained_vs_dropped.png', bbox_inches='tight')
plt.show()
print('Figure saved: fig6_distribution_retained_vs_dropped.png')

---
## 14. Step 10 — Final Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.35)

# ── Panel A: Missingness % by variable ────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
colours_a = [severity_colour(p) for p in miss_df['Missing (%)']]
ax_a.barh(miss_df['Variable'], miss_df['Missing (%)'],
          color=colours_a, edgecolor='white', alpha=0.85)
ax_a.axvline(MISSING_HIGH_PCT, color='red', linestyle='--', lw=1.3)
ax_a.set_title('A. Missingness by Variable', fontweight='bold')
ax_a.set_xlabel('Missing (%)')

# ── Panel B: Records with N missing ───────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
ax_b.bar(summary_dist['N variables missing'], summary_dist['% of dataset'],
         color=plt.cm.viridis(np.linspace(0.2, 0.9, len(summary_dist))),
         edgecolor='white', alpha=0.85)
ax_b.set_title('B. % Records by # Variables Missing', fontweight='bold')
ax_b.set_xlabel('# Variables Missing')
ax_b.set_ylabel('% of dataset')
ax_b.set_xticks(summary_dist['N variables missing'])

# ── Panel C: Listwise deletion waterfall ──────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
wf_labels = ['Raw\ndataset', 'After listwise\ndeletion']
wf_vals   = [n_total, n_after]
bars = ax_c.bar(wf_labels, wf_vals, color=['#1f77b4', '#2ca02c'],
                edgecolor='white', alpha=0.85, width=0.55)
for bar, val in zip(bars, wf_vals):
    ax_c.text(bar.get_x() + bar.get_width()/2,
              bar.get_height() + max(wf_vals)*0.01,
              f'{val:,}', ha='center', va='bottom', fontweight='bold')
ax_c.set_title(f'C. Listwise Deletion: {pct_lost:.2f}% lost', fontweight='bold')
ax_c.set_ylabel('Number of Records')

# ── Panel D: BW missingness by setting (if available) ─────────────────────────
ax_d = fig.add_subplot(gs[1, 1])
if BW_COL in df_raw.columns and HOSP_COL in df_raw.columns:
    colours_d = ['#d62728' if 'No' in str(s) or 'Not' in str(s) else '#2ca02c'
                  for s in bw_miss_table[HOSP_COL]]
    bars_d = ax_d.bar(bw_miss_table[HOSP_COL].astype(str),
                       bw_miss_table['Missing (%)'],
                       color=colours_d, edgecolor='white', alpha=0.85)
    for bar, pct in zip(bars_d, bw_miss_table['Missing (%)']):
        ax_d.text(bar.get_x() + bar.get_width()/2,
                  bar.get_height() + bw_miss_table['Missing (%)'].max()*0.02,
                  f'{pct:.2f}%', ha='center', va='bottom', fontweight='bold')
    ax_d.set_title('D. BW Missing % by Delivery Setting (MAR test)', fontweight='bold')
    ax_d.set_ylabel('% Missing')
else:
    ax_d.text(0.5, 0.5, 'Birth Weight or\nDelivery Setting\nnot available',
              ha='center', va='center', transform=ax_d.transAxes, fontsize=12)
    ax_d.set_xticks([])
    ax_d.set_yticks([])

fig.suptitle('Missing Data Analysis — Summary Dashboard\n'
             'Sri Lanka CRVS 2000 | Section 3.4.2 (Rubin, 1976; Little & Rubin, 2002)',
             fontweight='bold', fontsize=13)
plt.savefig('fig7_missing_data_dashboard.png', bbox_inches='tight')
plt.show()
print('Figure saved: fig7_missing_data_dashboard.png')

---
## 15. Step 11 — Export Cleaned Analytical Dataset

In [ ]:
# ── Drop the helper retention indicator ───────────────────────────────────────
df_complete = df_complete.drop(columns=['_retained'], errors='ignore')

# ── Export ────────────────────────────────────────────────────────────────────
OUTPUT_FILE = 'crvs_2000_complete_cases.csv'
df_complete.to_csv(OUTPUT_FILE, index=False)

print(f'Complete-case analytical dataset exported → {OUTPUT_FILE}')
print(f'Shape: {df_complete.shape}')
print()
print('This dataset is ready for:')
print('  • Outlier analysis (Section 3.4.1)')
print('  • Chi-square tests of association (Section 3.5.2)')
print('  • Regression modelling: binary, ordinal, multinomial (Section 3.5.3)')

# ── Export missingness summary tables ─────────────────────────────────────────
miss_df.to_csv('missingness_summary_by_variable.csv', index=False)
if mar_rows:
    pd.DataFrame(mar_rows).to_csv('mar_mechanism_tests.csv', index=False)
compare_df.to_csv('retained_vs_dropped_comparison.csv', index=False)

print()
print('Supporting tables exported:')
print('  • missingness_summary_by_variable.csv')
print('  • mar_mechanism_tests.csv')
print('  • retained_vs_dropped_comparison.csv')

---
## 16. Final Summary and Methodological Conclusions

In [ ]:
print('═══ MISSING DATA ANALYSIS — FINAL SUMMARY ════════════════════════════════')
print()
print(f'Raw dataset                              : {n_total:>10,} records')
print(f'Records retained after listwise deletion : {n_after:>10,} records')
print(f'Records dropped                          : {n_lost:>10,} records ({pct_lost:.2f}%)')
print()
print('── Variable-Level Summary ────────────────────────────────────────────────')
for _, row in miss_df.iterrows():
    print(f"  {row['Variable']:<25} : {row['Missing (%)']:>6.2f}% missing — {row['Severity']}")
print()
print('── Mechanism Assessment ──────────────────────────────────────────────────')
if 'mcar_result' in dir():
    p_mcar = mcar_result.get('p_value', np.nan)
    if not np.isnan(p_mcar):
        print(f"  Little's MCAR test p-value : "
              f"{'< 0.001' if p_mcar < 0.001 else f'{p_mcar:.4f}'}")
        print(f"  → {mcar_result['conclusion']}")
print()
if BW_COL in df_raw.columns and HOSP_COL in df_raw.columns:
    print('  Birth Weight × Delivery Setting (MAR test):')
    print(f"    χ² = {chi2_stat:.2f}, df = {dof}, "
          f"p = {'< 0.001' if p_val < 0.001 else f'{p_val:.4f}'}, V = {cramers_v_bw:.4f}")
    if p_val < ALPHA:
        print('    → MAR mechanism CONFIRMED — listwise deletion valid when delivery')
        print('      setting is included in regression models.')
print()
print('── Methodology Decision (Section 3.4.2.3) ────────────────────────────────')
print('  Approach     : Listwise deletion (complete-case analysis)')
print('  Justification: Large sample, low overall missingness, MAR assumption')
print('                 supported by delivery-setting analysis.')
print('  References   : Rubin (1976); Little & Rubin (2002); Sterne et al. (2009);')
print('                 Rutstein & Rojas (2006).')
print()
print('═══ END OF MISSING DATA ANALYSIS ══════════════════════════════════════════')

---
## 17. Full Reference List

All references cited in this notebook follow the methodology chapter (Chapter 3) of the study.

- **Agresti, A.** (2018). *Statistical methods for the social sciences* (5th ed.). Pearson Education.

- **Department of Census and Statistics Sri Lanka (DCS).** (2002). *Sri Lanka Demographic and Health Survey 2000*. https://www.statistics.gov.lk/Resource/en/Social/DHS/SLDHS2000.pdf

- **Hosmer, D. W., Lemeshow, S., & Sturdivant, R. X.** (2013). *Applied logistic regression* (3rd ed.). Wiley. https://doi.org/10.1002/9781118548387

- **Little, R. J. A.** (1988). A test of missing completely at random for multivariate data with missing values. *Journal of the American Statistical Association*, 83(404), 1198–1202. https://doi.org/10.1080/01621459.1988.10478722

- **Little, R. J. A., & Rubin, D. B.** (2002). *Statistical analysis with missing data* (2nd ed.). Wiley. https://doi.org/10.1002/9781119013563

- **Ministry of Health Sri Lanka.** (2000). *Annual Health Bulletin 2000*. Ministry of Health. https://www.health.gov.lk/moh_final/english/others.php?pid=71

- **Rubin, D. B.** (1976). Inference and missing data. *Biometrika*, 63(3), 581–592. https://doi.org/10.1093/biomet/63.3.581

- **Rutstein, S. O., & Rojas, G.** (2006). *Guide to DHS statistics* (DHS-IV). ORC Macro. https://www.dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_29Oct2012_DHSG1.pdf

- **Sterne, J. A. C., White, I. R., Carlin, J. B., Spratt, M., Royston, P., Kenward, M. G., Wood, A. M., & Carpenter, J. R.** (2009). Multiple imputation for missing data in epidemiological and clinical research: Potential and pitfalls. *BMJ*, 338, b2393. https://doi.org/10.1136/bmj.b2393

- **United Nations, Department of Economic and Social Affairs, Population Division.** (2014). *Principles and recommendations for a vital statistics system: Revision 3* (Statistical Papers, Series M, No. 19/Rev.3). United Nations. https://unstats.un.org/unsd/demographic/standmeth/principles/m19rev3en.pdf

- **World Health Organization.** (2014). *Global nutrition targets 2025: Low birth weight policy brief* (WHO/NMH/NHD/14.5). WHO. https://www.who.int/publications/i/item/WHO-NMH-NHD-14.5